# Sentinel-2 cloud-masked median composite via openEO

The `sentinel-2-l2a-cloud-masked-composite` recipe masks clouds (`mask_scl_dilation`) and reduces the time axis to a **median composite** (`reduce_dimension(t, median)`), returning a single multi-band GeoTIFF. Composites over a season are larger, so we run this as a **batch job** (`execute='batch'`), which the backend polls to completion.

```bash
pip install earthlens[openeo]
```

## Credentials

openEO authenticates with CDSE via OIDC. The download cell below runs for real when credentials are present (env `OPENEO_CLIENT_ID` / `OPENEO_CLIENT_SECRET`, or a cached refresh token from a prior interactive login) and prints a skip note otherwise - so this notebook executes cleanly with or without secrets. See the Authentication page.

In [ ]:
import os
from pathlib import Path


def has_openeo_credentials() -> bool:
    """Whether some openEO OIDC credential source is available."""
    if os.environ.get("OPENEO_CLIENT_ID") and os.environ.get("OPENEO_CLIENT_SECRET"):
        return True
    if os.environ.get("OPENEO_REFRESH_TOKEN"):
        return True
    return (Path.home() / ".openeo").exists()


def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note.

    Keeps the notebook executing top-to-bottom with no errors whether or not
    CDSE OIDC credentials are configured (so the docs build never needs secrets).
    """
    if not has_openeo_credentials():
        print(
            "No openEO OIDC credentials found - skipping the live download.\n"
            "Set OPENEO_CLIENT_ID / OPENEO_CLIENT_SECRET (headless) or sign in once\n"
            "interactively (see the Authentication page) to run this cell for real."
        )
        return []
    paths = facade.download(**download_kwargs)
    for path in paths:
        print(path)
    return paths


## Build the request

In [ ]:
from earthlens import EarthLens

facade = EarthLens(
    data_source='openeo',
    variables={'sentinel-2-l2a-cloud-masked-composite': []},
    start='2023-06-01', end='2023-08-31',
    lat_lim=[40.30, 40.50], lon_lim=[3.60, 3.80],
    path='data/openeo-composite',
    execute='batch',
    max_cloud_cover=60,
)
facade.datasource._output_format

## Download (server-side execution)

In [ ]:
paths = run_or_skip(facade)
paths